# ROGII E04: Heel-Calibrated Bimodal Datum Hedge

Every run is self-contained and uses only the mounted competition data.
The honest fallback is the previously validated residual LightGBM stack. Each
experiment changes one correction layer and writes standardized validation,
contact, prefix, feature-importance, and submission-integrity reports.

In [ ]:
# Control panel: this is the only visible code cell.
import os

SUBMISSION_PROFILE = "contact_gated_anchor"  # previous_stack | contact_gated_anchor

RIDGE_ANCHOR_WEIGHT = 0.30
SELECTOR_ANCHOR_WEIGHT = 0.70
SELECTOR_PF_WEIGHT = 0.85
PROJECTION_DEGREE = 4
PROJECTION_WEIGHT = 0.75
PROJECTED_ANCHOR_WEIGHT = 0.60

CONTACT_REFERENCE = "EGFDU"
CONTACT_PREFIX_RMSE_LIMIT = 1.0
CONTACT_MIN_PREFIX_ROWS = 50
CONTACT_MIN_PHYS_ROWS = 100

RUN_HEEL_DATUM_AUDIT = True
APPLY_BIMODAL_HEDGE = True
RUN_VISIBLE_PREFIX_AUDIT = True
PREFIX_CUT_FRACTIONS = (0.50, 0.65, 0.75)
PREFIX_MIN_GAIN = 1.0
PREFIX_MAX_BEST_RMSE = 9.0
PREFIX_MIN_CONSISTENCY = 0.67
PREFIX_MOVE_WEIGHT_CAP = 0.22
PREFIX_MOVE_CLIP = 18.0

TRAIN_PF_PARTICLES = 120
TRAIN_PF_SEEDS = 8
TEST_PF_PARTICLES = 500
TEST_PF_SEEDS = 48
N_JOBS = 4
LGB_ESTIMATORS = 1200
STACK_SHRINK = 0.95
STACK_CLIP = 60.0
RUN_GROUP_CV = False
CV_FOLDS = 5

# Local runs automatically use a small smoke subset. Kaggle always uses all wells.
LOCAL_DEBUG = not os.path.exists("/kaggle/input")
DEBUG_TRAIN_WELLS = 12

if SUBMISSION_PROFILE not in {"previous_stack", "contact_gated_anchor"}:
    raise ValueError("Unknown SUBMISSION_PROFILE")


In [ ]:
"""
Particle-filter geosteering tracker (numpy, vectorised over particles).

State per particle: (pos, rate) where pos = TVT + Z  ("structural" position that
is ~constant when geology is flat), rate = d(pos)/dMD. At each eval step we
propagate with momentum, convert to TVT = pos - Z, score against the typewell
GR(TVT) curve via a Gaussian likelihood, resample, and take the weighted-mean TVT.

Predictions are made only for the eval rows (TVT_input is NaN). A seed ensemble
is combined with weights ∝ exp(loglik/scale). This is the core signal that gets
geosteering solutions to single-digit RMSE.
"""
import numpy as np

# tuned defaults (close to strong public solutions)
MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5


def _run_pf(md, z, gr, tw_tvt, tw_gr, last_tvt, last_Z, ir, gs,
            n_particles=500, seed=0):
    rng = np.random.default_rng(seed)
    N = n_particles
    ls = last_tvt + last_Z
    pos = ls + 2.0 * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N) / N
    lo, hi = tw_tvt[0] - 100, tw_tvt[-1] + 100
    out = np.empty(len(md)); loglik = 0.0
    prev_md = md[0] - 1.0
    for i in range(len(md)):
        dm = max(md[i] - prev_md, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos = pos + rate * dm + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - z[i], lo, hi)
        pos = tvt_p + z[i]
        if np.isfinite(gr[i]):
            eg = np.interp(tvt_p, tw_tvt, tw_gr)
            d = (gr[i] - eg) / gs
            lk = np.maximum(np.exp(-0.5 * np.minimum(d * d, 600.0)), 1e-300)
            avg = float((w * lk).sum()); loglik += np.log(max(avg, 1e-300))
            w = w * lk; s = w.sum(); w = w / s if s > 0 else np.ones(N) / N
        neff = 1.0 / (w * w).sum()
        if neff < RESAMP * N:
            cum = np.cumsum(w); u0 = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos = pos[idx] + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w = np.ones(N) / N
        out[i] = float(np.dot(w, pos - z[i]))
        prev_md = md[i]
    return out, loglik


def pf_predict(hw, tw, n_particles=500, n_seeds=48, scale=5.0):
    """Return a full-length TVT array (known rows kept, eval rows predicted)."""
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)

    kn = hw[hw["TVT_input"].notna()]
    ev = hw[hw["TVT_input"].isna()]
    out = hw["TVT_input"].to_numpy(float).copy()
    if len(ev) == 0:
        return out

    last = kn.iloc[-1]
    last_tvt = float(last["TVT_input"]); last_Z = float(last["Z"])
    # GR noise sigma from known section
    tw_at_k = np.interp(kn["TVT_input"].to_numpy(), tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn["GR"].fillna(0).to_numpy() - tw_at_k), 10., 60.))
    # initial structural rate from last 30 known points
    tail = kn.tail(30)
    dt = np.diff(tail["TVT_input"].to_numpy()); dz = np.diff(tail["Z"].to_numpy())
    dmv = np.diff(tail["MD"].to_numpy()); m = dmv > 0
    ir = float(np.median((dt + dz)[m] / dmv[m])) if m.sum() >= 3 else 0.0

    gr_all = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    md = ev["MD"].to_numpy(float); z = ev["Z"].to_numpy(float)
    gr = gr_all[ev.index.to_numpy()]
    # prepend last known so momentum starts from the right place
    md = np.concatenate([[float(last["MD"])], md])
    z = np.concatenate([[last_Z], z])
    gr = np.concatenate([[np.nan], gr])

    preds = np.empty((n_seeds, len(md))); liks = np.empty(n_seeds)
    for s in range(n_seeds):
        preds[s], liks[s] = _run_pf(md, z, gr, tw_tvt, tw_gr, last_tvt, last_Z,
                                    ir, gs, n_particles, seed=s)
    liks -= liks.max()
    wts = np.exp(liks / scale); wts /= wts.sum()
    est = (wts[:, None] * preds).sum(0)[1:]        # drop the prepended known row
    out[ev.index.to_numpy()] = est
    return out


In [ ]:
"""Beam-search geosteering tracker (numba JIT). Complements the particle filter:
finds a low-cost monotone-ish path through the typewell GR(TVT) grid that matches
the horizontal GR signature. Ensembled over several stiffness configs."""
import numpy as np
from numba import njit
from scipy.signal import savgol_filter

# (beam_size, move_cost, err_scale, savgol_radius)
BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2), (10, 8.0, 64.0, 2), (8, 35.0, 220.0, 1),
    (10, 14.0, 90.0, 5), (20, 4.0, 36.0, 3), (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
]


@njit(cache=False)
def _beam(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS * 6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30); bcost[0] = 0.0; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt:
                    continue
                tot = cost + (gv - tw_gr[ni]) ** 2 / es + mc * (d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni:
                        fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]:
                        cC[fnd] = tot; cP[fnd] = bi
                elif nc < MAX:
                    cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i + 1, nc):
                if cC[j] < cC[mi]:
                    mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]
                cC[i], cC[mi] = cC[mi], cC[i]
                cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]:
            best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n - 1, -1, -1):
        path[s] = hI[s, b]; b = hP[s, b]
    return path


def beam_predict(hw, tw):
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    out = hw["TVT_input"].to_numpy(float).copy()
    if len(ev) == 0:
        return out
    last_tvt = float(kn.iloc[-1]["TVT_input"])
    gr_all = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    hgr = gr_all[ev.index.to_numpy()]
    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))
    paths = []
    for BS, mc, es, r in BEAM_CONFIGS:
        if r > 0 and len(hgr) > max(3, 2 * r + 1):
            win = min(2 * r + 1, len(hgr) if len(hgr) % 2 == 1 else len(hgr) - 1)
            sgr = savgol_filter(hgr, win, min(2, win - 1))
        else:
            sgr = hgr.copy()
        paths.append(tw_tvt[_beam(sgr.astype(np.float64), tw_gr, si, BS, float(mc), float(es))])
    out[ev.index.to_numpy()] = np.stack(paths, 0).mean(0)
    return out


In [ ]:
def ps_index(h):
    return int(h['TVT_input'].notna().sum())

"""
Compute honest domain signals per well (post-PS rows) for the LightGBM residual
stack. Target-free (no hidden TVT used). Saves parquet for train and test.

Signals: PF path (delta), beam path (delta), multi-scale NCC vs typewell,
self-correlation vs the lateral's own known section, Q-3D tortuosity, relative
trajectory geometry, GR texture, typewell GR residuals at PF/anchor baselines.
Target: dTVT = TVT - last_known_TVT (train only).
"""
import sys, os, glob, time
import numpy as np
import pandas as pd
from joblib import Parallel, delayed






ANCH_OFF = [-20, -10, -5, 0, 5, 10, 20]


def multiscale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    out = []
    for hw in hws:
        win = 2 * hw + 1; nk = len(kgr); nh = len(hgr)
        if nk < win + 1 or nh == 0:
            out.append((np.full(nh, ktvt[-1] if len(ktvt) else 0.0, np.float32), np.zeros(nh, np.float32)))
            continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().to_numpy(np.float32)
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().to_numpy(np.float32)
        sts = np.arange(0, nk - win + 1, stride, dtype=np.int32); M = len(sts)
        if M == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        C = kg[sts[:, None] + np.arange(win, dtype=np.int32)[None, :]]
        Cn = (C - C.mean(1, keepdims=True)) / (C.std(1, keepdims=True) + 1e-6)
        hp = np.pad(hg, hw, mode="edge")
        Hm = hp[np.arange(nh)[:, None] + np.arange(win)[None, :]]
        Hn = (Hm - Hm.mean(1, keepdims=True)) / (Hm.std(1, keepdims=True) + 1e-6)
        ncc = Hn @ Cn.T / win
        best = ncc.argmax(1); score = ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best] + hw, 0, nk - 1)].astype(np.float32), score))
    return out


def tortuosity(x, y, z, md, win=50):
    d = np.stack([np.gradient(x), np.gradient(y), np.gradient(z)], 1)
    dm = np.gradient(md)[:, None] + 1e-9
    t = d / dm
    n = np.linalg.norm(t, axis=1, keepdims=True) + 1e-9
    u = t / n
    kappa = np.linalg.norm(np.gradient(u, axis=0), axis=1)   # curvature proxy
    return pd.Series(kappa).rolling(win, min_periods=1).sum().to_numpy(np.float32)


def build_well(path, is_train):
    wid = os.path.basename(path).split("__")[0]
    base = os.path.dirname(path)
    try:
        h = pd.read_csv(path)
        tw = pd.read_csv(f"{base}/{wid}__typewell.csv")
    except Exception:
        return None
    ps = ps_index(h)
    if ps < 10 or ps >= len(h) - 3:
        return None
    if is_train and "TVT" not in h.columns:
        return None
    tw_s = tw.sort_values("TVT"); tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)
    if len(tw_tvt) < 3:
        return None

    tps = float(h["TVT_input"].iloc[ps - 1])
    md = h["MD"].to_numpy(float); x = h["X"].to_numpy(float); y = h["Y"].to_numpy(float); z = h["Z"].to_numpy(float)
    gr_all = h["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    ev = np.arange(ps, len(h)); nev = len(ev)

    pf = pf_predict(h, tw, n_particles=TRAIN_PF_PARTICLES if is_train else TEST_PF_PARTICLES, n_seeds=TRAIN_PF_SEEDS if is_train else TEST_PF_SEEDS, scale=12.0)
    bm = beam_predict(h, tw)
    ktvt = h["TVT_input"].to_numpy(float)[:ps]
    kgr = gr_all[:ps]; hgr = gr_all[ps:]
    ncc = multiscale_ncc(kgr, ktvt, hgr)
    tort = tortuosity(x, y, z, md)

    md_ps, x_ps, y_ps, z_ps = md[ps - 1], x[ps - 1], y[ps - 1], z[ps - 1]
    gr_s = pd.Series(gr_all)
    f = {
        "well": wid, "id": [f"{wid}_{i}" for i in ev],
        "last_known_tvt": tps,
        "pf_d": (pf[ev] - tps).astype(np.float32),
        "beam_d": (bm[ev] - tps).astype(np.float32),
        "pf_vs_beam": (pf[ev] - bm[ev]).astype(np.float32),
        "ncc8_d": np.clip(ncc[0][0] - tps, -80, 80), "ncc8_s": ncc[0][1],
        "ncc15_d": np.clip(ncc[1][0] - tps, -80, 80), "ncc15_s": ncc[1][1],
        "ncc25_d": np.clip(ncc[2][0] - tps, -80, 80), "ncc25_s": ncc[2][1],
        "ncc_mean_d": np.clip((ncc[0][0] + ncc[1][0] + ncc[2][0]) / 3 - tps, -80, 80),
        "tort": tort[ev],
        "d_md": (md[ev] - md_ps).astype(np.float32),
        "d_z": (z[ev] - z_ps).astype(np.float32),
        "d_xy": np.hypot(x[ev] - x_ps, y[ev] - y_ps).astype(np.float32),
        "dz_dmd": (np.gradient(z) / (np.gradient(md) + 1e-9))[ev].astype(np.float32),
        "gr": gr_all[ev].astype(np.float32),
        "gr_m21": gr_s.rolling(21, center=True, min_periods=1).mean().to_numpy()[ev].astype(np.float32),
        "gr_s21": gr_s.rolling(21, center=True, min_periods=1).std().fillna(0).to_numpy()[ev].astype(np.float32),
        "gr_d1": gr_s.diff().fillna(0).to_numpy()[ev].astype(np.float32),
        "frac": (np.arange(nev) / max(nev - 1, 1)).astype(np.float32),
    }
    for o in ANCH_OFF:
        f[f"twres_pf{o}"] = (gr_all[ev] - np.interp(pf[ev] + o, tw_tvt, tw_gr)).astype(np.float32)
    if is_train:
        f["target"] = (h["TVT"].to_numpy(float)[ev] - tps).astype(np.float32)
    return pd.DataFrame(f)




In [ ]:
import glob
import hashlib
import json
import math
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.signal import savgol_filter
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")


def find_data_root():
    candidates = [
        "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
        "/kaggle/input/rogii-wellbore-geology-prediction",
        "data",
    ] + sorted(glob.glob("/kaggle/input/*"))
    for root in candidates:
        if (
            os.path.exists(f"{root}/sample_submission.csv")
            and glob.glob(f"{root}/train/*__horizontal_well.csv")
        ):
            return Path(root)
    hits = glob.glob("/kaggle/input/**/*__horizontal_well.csv", recursive=True)
    if hits:
        return Path(hits[0]).parent.parent
    raise FileNotFoundError("Could not locate the competition data")


DATA = find_data_root()
print("data root:", DATA)
print("local debug:", LOCAL_DEBUG)


def build_split(split, is_train):
    paths = sorted(glob.glob(str(DATA / split / "*__horizontal_well.csv")))
    if LOCAL_DEBUG and is_train:
        paths = paths[:DEBUG_TRAIN_WELLS]
    results = Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(build_well)(path, is_train) for path in paths
    )
    frames = [frame for frame in results if frame is not None and len(frame)]
    if not frames:
        raise RuntimeError(f"No usable {split} wells")
    return pd.concat(frames, ignore_index=True)


print("building target-free test signals")
test = build_split("test", False)
print("building train signals")
train = build_split("train", True)
train["target_surface"] = train["target"].astype(float) + train["d_z"].astype(float)
print(
    "train rows/wells:", len(train), train["well"].nunique(),
    "| test rows/wells:", len(test), test["well"].nunique(),
)

DROP = {"well", "id", "target", "target_surface", "last_known_tvt"}
FEATURES = [column for column in train.columns if column not in DROP]
X = np.nan_to_num(
    train[FEATURES].to_numpy(np.float32), nan=0.0, posinf=0.0, neginf=0.0
)
y = train["target"].to_numpy(np.float32)
groups = train["well"].to_numpy()
Xt = np.nan_to_num(
    test[FEATURES].to_numpy(np.float32), nan=0.0, posinf=0.0, neginf=0.0
)


def pooled_rmse(actual, predicted):
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(predicted)) ** 2)))


def mean_well_rmse(frame, actual, predicted):
    audit = pd.DataFrame(
        {
            "well": frame["well"].to_numpy(),
            "sse": (np.asarray(actual) - np.asarray(predicted)) ** 2,
        }
    )
    return float(audit.groupby("well")["sse"].mean().pow(0.5).mean())


# Ridge is deliberately sampled: it is a broad, regularized anchor, not the final learner.
rng = np.random.default_rng(2026)
ridge_n = min(len(train), 500_000)
ridge_idx = rng.choice(len(train), size=ridge_n, replace=False)
scaler = StandardScaler().fit(X[ridge_idx])
ridge_surface_y = train["target_surface"].to_numpy(np.float32)
ridge = Ridge(alpha=30.0).fit(
    scaler.transform(X[ridge_idx]), ridge_surface_y[ridge_idx]
)
ridge_test_ds = ridge.predict(scaler.transform(Xt))

LGB_PARAMS = dict(
    objective="regression",
    n_estimators=LGB_ESTIMATORS if not LOCAL_DEBUG else 120,
    learning_rate=0.02,
    num_leaves=127,
    min_child_samples=100,
    subsample=0.80,
    subsample_freq=1,
    colsample_bytree=0.70,
    reg_lambda=5.0,
    reg_alpha=1.0,
    verbose=-1,
    n_jobs=-1,
    random_state=2026,
)

cv_rows = []
if RUN_GROUP_CV and train["well"].nunique() >= CV_FOLDS:
    oof = np.zeros(len(train), dtype=float)
    splitter = GroupKFold(n_splits=CV_FOLDS)
    for fold, (fit_idx, valid_idx) in enumerate(splitter.split(X, y, groups)):
        model = lgb.LGBMRegressor(**LGB_PARAMS).fit(X[fit_idx], y[fit_idx])
        oof[valid_idx] = model.predict(X[valid_idx])
        fold_rmse = pooled_rmse(y[valid_idx], oof[valid_idx])
        print("CV fold", fold, "pooled residual-stack RMSE", round(fold_rmse, 4))
    cv_rows.append(
        {
            "candidate": "previous_residual_stack",
            "pooled_rmse": pooled_rmse(y, oof),
            "mean_well_rmse": mean_well_rmse(train, y, oof),
        }
    )
else:
    split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=2026)
    fit_idx, valid_idx = next(split.split(X, y, groups))
    probe = lgb.LGBMRegressor(**LGB_PARAMS).fit(X[fit_idx], y[fit_idx])
    probe_pred = probe.predict(X[valid_idx])
    cv_rows.append(
        {
            "candidate": "previous_residual_stack_holdout",
            "pooled_rmse": pooled_rmse(y[valid_idx], probe_pred),
            "mean_well_rmse": mean_well_rmse(
                train.iloc[valid_idx], y[valid_idx], probe_pred
            ),
        }
    )

model = lgb.LGBMRegressor(**LGB_PARAMS).fit(X, y)
learned_test_d = np.clip(
    model.predict(Xt) * STACK_SHRINK, -STACK_CLIP, STACK_CLIP
)
pd.DataFrame(cv_rows).to_csv(WORK / "training_validation_report.csv", index=False)
pd.Series(model.feature_importances_, index=FEATURES).sort_values(
    ascending=False
).rename("importance").to_csv(WORK / "feature_importance.csv")


def smooth(values):
    values = np.asarray(values, dtype=float)
    n = len(values)
    window = min(31, n if n % 2 else n - 1)
    if window >= 5:
        return savgol_filter(values, window, min(3, window - 1))
    return values


test["ridge_ds"] = ridge_test_ds
test["learned_d"] = learned_test_d
test["tvt_ridge"] = (
    test["last_known_tvt"].astype(float) + test["ridge_ds"] - test["d_z"]
)
test["tvt_learned"] = (
    test["last_known_tvt"].astype(float) + test["learned_d"]
)
test["tvt_pf"] = test["last_known_tvt"].astype(float) + test["pf_d"].astype(float)
test["tvt_beam"] = (
    test["last_known_tvt"].astype(float) + test["beam_d"].astype(float)
)
test["tvt_selector"] = (
    SELECTOR_PF_WEIGHT * test["tvt_pf"]
    + (1.0 - SELECTOR_PF_WEIGHT) * test["tvt_beam"]
)
test["tvt_anchor"] = (
    RIDGE_ANCHOR_WEIGHT * test["tvt_ridge"]
    + SELECTOR_ANCHOR_WEIGHT * test["tvt_selector"]
)


def robust_polyfit(x, y_values, degree):
    x = np.asarray(x, dtype=float)
    y_values = np.asarray(y_values, dtype=float)
    good = np.isfinite(x) & np.isfinite(y_values)
    if good.sum() < max(6, degree + 2):
        return y_values.copy()
    xx, yy = x[good], y_values[good]
    deg = min(int(degree), int(good.sum()) - 2)
    coef = np.polyfit(xx, yy, deg)
    for _ in range(5):
        residual = yy - np.polyval(coef, xx)
        scale = 1.4826 * np.median(np.abs(residual - np.median(residual))) + 1e-6
        weight = 1.0 / (1.0 + (residual / (2.0 * scale)) ** 2)
        coef = np.polyfit(xx, yy, deg, w=weight)
    output = y_values.copy()
    output[good] = np.polyval(coef, xx)
    return output


def load_pair(well, split):
    base = DATA / split
    return (
        pd.read_csv(base / f"{well}__horizontal_well.csv"),
        pd.read_csv(base / f"{well}__typewell.csv"),
    )


def project_group(group):
    group = group.copy()
    well = str(group["well"].iloc[0])
    hw, _ = load_pair(well, "test")
    row_idx = group["id"].str.rsplit("_", n=1).str[-1].astype(int).to_numpy()
    md = hw["MD"].to_numpy(float)[row_idx]
    z = hw["Z"].to_numpy(float)[row_idx]
    known = hw[hw["TVT_input"].notna()]
    last = known.iloc[-1]
    anchor_u = float(last["TVT_input"]) + float(last["Z"])
    start_md = float(last["MD"])
    denom = max(float(hw["MD"].iloc[-1]) - start_md, 1e-6)
    s = (md - start_md) / denom
    raw = group["tvt_anchor"].to_numpy(float)
    delta_u = raw + z - anchor_u
    fitted_delta_u = robust_polyfit(s, delta_u, PROJECTION_DEGREE)
    projected = anchor_u + fitted_delta_u - z
    group["tvt_projected"] = (
        (1.0 - PROJECTION_WEIGHT) * raw + PROJECTION_WEIGHT * projected
    )
    # Controlled ablation: the previous residual stack is the complete
    # non-contact fallback. Projection remains available for diagnostics.
    group["tvt_honest"] = smooth(group["tvt_learned"].to_numpy(float))
    return group


test = pd.concat(
    [project_group(group) for _, group in test.groupby("well", sort=False)],
    ignore_index=True,
)


def heel_calibration_and_scan(well, base_pred):
    hw, tw = load_pair(well, "test")
    known = hw[hw["TVT_input"].notna()].copy()
    tws = tw.sort_values("TVT")
    tw_tvt = tws["TVT"].to_numpy(float)
    tw_gr = tws["GR"].interpolate(limit_direction="both").to_numpy(float)
    expected = np.interp(known["TVT_input"].to_numpy(float), tw_tvt, tw_gr)
    observed = known["GR"].to_numpy(float)
    mask = np.isfinite(expected) & np.isfinite(observed)
    if mask.sum() < 40:
        return base_pred, {"well": well, "status": "too_few_prefix_rows"}
    design = np.c_[expected[mask], np.ones(mask.sum())]
    alpha, beta = np.linalg.lstsq(design, observed[mask], rcond=None)[0]
    alpha = float(np.clip(alpha, 0.35, 2.50))
    beta = float(np.clip(beta, -120.0, 120.0))
    calibrated = (hw["GR"].to_numpy(float) - beta) / alpha
    eval_idx = np.flatnonzero(hw["TVT_input"].isna().to_numpy())
    gr_eval = calibrated[eval_idx]
    valid = np.isfinite(gr_eval) & np.isfinite(base_pred)
    if valid.sum() < 80:
        return base_pred, {
            "well": well,
            "status": "too_few_hidden_gr_rows",
            "alpha": alpha,
            "beta": beta,
        }
    prefix_resid = observed[mask] - (alpha * expected[mask] + beta)
    scale = float(np.clip(1.4826 * np.median(np.abs(prefix_resid)), 5.0, 60.0))
    shifts = np.arange(-20.0, 20.01, 0.5)
    scores = []
    for shift in shifts:
        residual = (
            gr_eval[valid]
            - np.interp(base_pred[valid] + shift, tw_tvt, tw_gr)
        ) / scale
        scores.append(float(np.mean(np.clip(residual, -6.0, 6.0) ** 2)))
    scores = np.asarray(scores)
    order = np.argsort(scores)
    first = int(order[0])
    second = next(
        (int(index) for index in order[1:] if abs(shifts[index] - shifts[first]) >= 8.0),
        int(order[min(1, len(order) - 1)]),
    )
    plausible = scores[second] <= scores[first] * 1.15
    fitted = alpha * expected[mask] + beta
    r2 = 1.0 - np.sum((observed[mask] - fitted) ** 2) / (
        np.sum((observed[mask] - observed[mask].mean()) ** 2) + 1e-9
    )
    trust = float(np.clip(r2, 0.0, 1.0))
    logits = -np.array([scores[first], scores[second]]) / 0.75
    logits -= logits.max()
    p_scan = float(np.exp(logits[0]) / np.exp(logits).sum())
    p_effective = trust * p_scan + (1.0 - trust) * 0.5
    hedge_shift = float(
        p_effective * shifts[first] + (1.0 - p_effective) * shifts[second]
    )
    output = base_pred + hedge_shift if APPLY_BIMODAL_HEDGE and plausible else base_pred
    return output, {
        "well": well,
        "status": "hedged" if APPLY_BIMODAL_HEDGE and plausible else "audit_only",
        "alpha": alpha,
        "beta": beta,
        "prefix_r2": trust,
        "shift_1": float(shifts[first]),
        "score_1": float(scores[first]),
        "shift_2": float(shifts[second]),
        "score_2": float(scores[second]),
        "plausible_bimodal": bool(plausible),
        "p_scan": p_scan,
        "p_effective": p_effective,
        "hedge_shift": hedge_shift,
    }


datum_rows = []
if RUN_HEEL_DATUM_AUDIT:
    adjusted = []
    for well, group in test.groupby("well", sort=False):
        values, report = heel_calibration_and_scan(
            well, group["tvt_honest"].to_numpy(float)
        )
        group = group.copy()
        group["tvt_honest"] = values
        adjusted.append(group)
        datum_rows.append(report)
    test = pd.concat(adjusted, ignore_index=True)
pd.DataFrame(datum_rows).to_csv(WORK / "heel_bimodal_audit.csv", index=False)


def prefix_candidate(hw, cut, name, target_idx):
    known_idx = np.flatnonzero(hw["TVT_input"].notna().to_numpy())
    train_idx = known_idx[:cut]
    if len(train_idx) < 20:
        return None
    tvt = hw["TVT_input"].to_numpy(float)
    z = hw["Z"].to_numpy(float)
    md = hw["MD"].to_numpy(float)
    last = train_idx[-1]
    if name == "flat":
        return np.full(len(target_idx), tvt[last])
    if name == "structural":
        return tvt[last] + z[last] - z[target_idx]
    tail = train_idx[-min(80, len(train_idx)) :]
    surface = tvt[tail] + z[tail]
    slope = np.polyfit(md[tail] - md[last], surface, 1)[0]
    slope = float(np.clip(slope, -0.08, 0.08))
    return tvt[last] + z[last] + slope * (md[target_idx] - md[last]) - z[target_idx]


def run_prefix_audit(well):
    hw, _ = load_pair(well, "test")
    known_idx = np.flatnonzero(hw["TVT_input"].notna().to_numpy())
    names = ("flat", "structural", "surface_trend")
    rows = []
    for fraction in PREFIX_CUT_FRACTIONS:
        cut = int(round(len(known_idx) * fraction))
        hold_idx = known_idx[cut:]
        if cut < 20 or len(hold_idx) < 10:
            continue
        truth = hw["TVT_input"].to_numpy(float)[hold_idx]
        for name in names:
            pred = prefix_candidate(hw, cut, name, hold_idx)
            rows.append(
                {
                    "well": well,
                    "cut_fraction": fraction,
                    "candidate": name,
                    "rmse": pooled_rmse(truth, pred),
                }
            )
    return rows


prefix_rows = []
if RUN_VISIBLE_PREFIX_AUDIT:
    for well in test["well"].drop_duplicates():
        prefix_rows.extend(run_prefix_audit(well))
prefix_report = pd.DataFrame(prefix_rows)
prefix_report.to_csv(WORK / "visible_prefix_audit.csv", index=False)


def apply_prefix_bounded(group):
    if SUBMISSION_PROFILE != "visible_prefix_bounded" or prefix_report.empty:
        return group["tvt_honest"].to_numpy(float), {
            "well": group["well"].iloc[0],
            "accepted": False,
            "reason": "profile_audit_only",
        }
    well = str(group["well"].iloc[0])
    report = prefix_report[prefix_report["well"] == well]
    if report.empty:
        return group["tvt_honest"].to_numpy(float), {
            "well": well, "accepted": False, "reason": "no_prefix_report"
        }
    summary = report.groupby("candidate")["rmse"].agg(["median", "std", "count"])
    summary["score"] = summary["median"] + 0.10 * summary["std"].fillna(0.0)
    winner = str(summary["score"].idxmin())
    best = float(summary.loc[winner, "score"])
    flat = float(summary.loc["flat", "score"])
    gain = flat - best
    cuts = report[report["candidate"] == winner]
    flat_by_cut = report[report["candidate"] == "flat"].set_index("cut_fraction")["rmse"]
    consistency = float(
        np.mean(
            [
                row.rmse < flat_by_cut.get(row.cut_fraction, np.inf)
                for row in cuts.itertuples()
            ]
        )
    )
    accepted = (
        winner != "flat"
        and gain >= PREFIX_MIN_GAIN
        and best <= PREFIX_MAX_BEST_RMSE
        and consistency >= PREFIX_MIN_CONSISTENCY
    )
    if not accepted:
        return group["tvt_honest"].to_numpy(float), {
            "well": well,
            "accepted": False,
            "reason": "quality_gate",
            "winner": winner,
            "gain": gain,
            "best_rmse": best,
            "consistency": consistency,
        }
    hw, _ = load_pair(well, "test")
    hidden_idx = group["id"].str.rsplit("_", n=1).str[-1].astype(int).to_numpy()
    known_count = int(hw["TVT_input"].notna().sum())
    candidate = prefix_candidate(hw, known_count, winner, hidden_idx)
    base = group["tvt_honest"].to_numpy(float)
    ramp = 1.0 - np.exp(-np.arange(len(base)) / max(80.0, 0.12 * len(base)))
    weight = min(PREFIX_MOVE_WEIGHT_CAP, 0.08 + 0.12 * min(gain, 5.0) / 5.0)
    move = np.clip(weight * ramp * (candidate - base), -PREFIX_MOVE_CLIP, PREFIX_MOVE_CLIP)
    return base + move, {
        "well": well,
        "accepted": True,
        "reason": "prefix_verified",
        "winner": winner,
        "gain": gain,
        "best_rmse": best,
        "consistency": consistency,
        "weight": weight,
        "max_abs_move": float(np.max(np.abs(move))),
    }


prefix_move_rows = []
prefix_adjusted = []
for _, group in test.groupby("well", sort=False):
    values, report = apply_prefix_bounded(group)
    group = group.copy()
    group["tvt_pre_contact"] = values
    prefix_adjusted.append(group)
    prefix_move_rows.append(report)
test = pd.concat(prefix_adjusted, ignore_index=True)
pd.DataFrame(prefix_move_rows).to_csv(WORK / "visible_prefix_moves.csv", index=False)


def contact_reconstruction(hw_train, tw_train, reference):
    if reference not in hw_train.columns:
        return None
    geology = tw_train.dropna(subset=["Geology", "TVT"]).copy()
    ref = geology.loc[geology["Geology"].astype(str) == reference, "TVT"]
    if ref.empty:
        return None
    ref_tvt = float(ref.min())
    raw = ref_tvt - (
        hw_train["Z"].to_numpy(float) - hw_train[reference].to_numpy(float)
    )
    bias = float(np.nanmean(hw_train["TVT"].to_numpy(float) - raw))
    result = raw + bias
    return result if np.isfinite(result).sum() >= CONTACT_MIN_PHYS_ROWS else None


def apply_contact_guard(group):
    well = str(group["well"].iloc[0])
    train_hw_path = DATA / "train" / f"{well}__horizontal_well.csv"
    train_tw_path = DATA / "train" / f"{well}__typewell.csv"
    base = group["tvt_pre_contact"].to_numpy(float)
    report = {
        "well": well,
        "accepted": False,
        "reference": CONTACT_REFERENCE,
        "replaced_rows": 0,
    }
    if not train_hw_path.exists() or not train_tw_path.exists():
        report["reason"] = "no_same_well_train_copy"
        return base, report
    hw_test, _ = load_pair(well, "test")
    hw_train = pd.read_csv(train_hw_path)
    tw_train = pd.read_csv(train_tw_path)
    contact = contact_reconstruction(hw_train, tw_train, CONTACT_REFERENCE)
    if contact is None:
        report["reason"] = "contact_unavailable"
        return base, report
    train_md = hw_train["MD"].to_numpy(float)
    known = hw_test[hw_test["TVT_input"].notna()].copy()
    in_range = known["MD"].between(np.nanmin(train_md), np.nanmax(train_md)).to_numpy()
    comparable = known.loc[in_range]
    if len(comparable) < CONTACT_MIN_PREFIX_ROWS:
        report["reason"] = "too_few_comparable_prefix_rows"
        report["prefix_rows"] = int(len(comparable))
        return base, report
    prefix_pred = np.interp(comparable["MD"].to_numpy(float), train_md, contact)
    prefix_truth = comparable["TVT_input"].to_numpy(float)
    prefix_rmse = pooled_rmse(prefix_truth, prefix_pred)
    report["prefix_rows"] = int(len(comparable))
    report["prefix_rmse"] = prefix_rmse
    if prefix_rmse > CONTACT_PREFIX_RMSE_LIMIT:
        report["reason"] = "prefix_rmse_failed"
        return base, report
    hidden_idx = group["id"].str.rsplit("_", n=1).str[-1].astype(int).to_numpy()
    hidden_md = hw_test["MD"].to_numpy(float)[hidden_idx]
    replace = (hidden_md >= np.nanmin(train_md)) & (hidden_md <= np.nanmax(train_md))
    output = base.copy()
    output[replace] = np.interp(hidden_md[replace], train_md, contact)
    report.update(
        {
            "accepted": True,
            "reason": "prefix_verified",
            "replaced_rows": int(replace.sum()),
        }
    )
    return output, report


contact_rows = []
contact_adjusted = []
for _, group in test.groupby("well", sort=False):
    values, report = apply_contact_guard(group)
    group = group.copy()
    group["tvt_contact"] = values
    contact_adjusted.append(group)
    contact_rows.append(report)
test = pd.concat(contact_adjusted, ignore_index=True)
pd.DataFrame(contact_rows).to_csv(WORK / "guarded_contact_report.csv", index=False)


sample = pd.read_csv(DATA / "sample_submission.csv")


def ordered_submission(column):
    prediction = test[["id", column]].rename(columns={column: "tvt"})
    if prediction["id"].duplicated().any():
        raise RuntimeError(f"Duplicate ids in {column}")
    output = sample[["id"]].merge(prediction, on="id", how="left", validate="one_to_one")
    if output["tvt"].isna().any():
        missing = output.loc[output["tvt"].isna(), "id"].head().tolist()
        raise RuntimeError(f"Missing predictions for {column}: {missing}")
    if output["id"].tolist() != sample["id"].tolist():
        raise RuntimeError("Submission id order changed")
    if not np.isfinite(output["tvt"].to_numpy(float)).all():
        raise RuntimeError(f"Non-finite predictions in {column}")
    return output


honest_sub = ordered_submission("tvt_pre_contact")
contact_sub = ordered_submission("tvt_contact")
honest_sub.to_csv(WORK / "submission_honest.csv", index=False)
contact_sub.to_csv(WORK / "submission_contact_gated.csv", index=False)
final_sub = honest_sub if SUBMISSION_PROFILE == "previous_stack" else contact_sub
final_sub.to_csv(WORK / "submission.csv", index=False)

sha = hashlib.sha256((WORK / "submission.csv").read_bytes()).hexdigest()
audit = {
    "profile": SUBMISSION_PROFILE,
    "fallback": "previous_residual_stack",
    "stack_shrink": float(STACK_SHRINK),
    "stack_clip": float(STACK_CLIP),
    "rows": int(len(final_sub)),
    "unique_ids": int(final_sub["id"].nunique()),
    "id_order_matches_sample": bool(final_sub["id"].tolist() == sample["id"].tolist()),
    "finite_tvt": bool(np.isfinite(final_sub["tvt"].to_numpy(float)).all()),
    "tvt_min": float(final_sub["tvt"].min()),
    "tvt_max": float(final_sub["tvt"].max()),
    "contact_wells_accepted": int(sum(bool(row.get("accepted")) for row in contact_rows)),
    "contact_rows_replaced": int(sum(int(row.get("replaced_rows", 0)) for row in contact_rows)),
    "sha256": sha,
}
(WORK / "submission_audit.json").write_text(json.dumps(audit, indent=2))
print(json.dumps(audit, indent=2))
print("wrote", WORK / "submission.csv")
final_sub.head()
